# 02 · Preparación de Datos

**Objetivo:** Construir un pipeline de preprocesado reutilizable para imágenes y datos tabulares.

**Datos de entrada:** `../data/raw/HAM10000_metadata.csv`, `../data/raw/hnmist_28_28_RGB.csv`

**Resultado esperado:** Conjuntos `X_img_train/test`, `X_tab_train/test`, `y_train/test` listos para entrenar cualquiera de los modelos.

> **Nota:** Las celdas de instalación solo es necesario ejecutarlas una vez.

## Instalación de dependencias

In [26]:
#El primer punto es instalar las librerías necesarias:
!pip install pandas 
!pip install scikit-learn
!pip install tensorflow
!pip install matplotlib
!pip install seaborn
!pip install keras
!pip install numpy
!pip install opencv-python
!pip install torch
!pip install torchvision


In [27]:
# Dado que tengo una GPU NVIDIA, instalo PyTorch con soporte CUDA para hacer pruebas con esta GPU:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121


In [28]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.9.1+cpu
None
False


## Descarga del dataset

Descarga los datos desde Google Drive. Si ya tienes los CSVs en `data/raw/`, salta esta celda.

In [29]:
!pip install gdown 

In [30]:
!gdown --folder https://drive.google.com/drive/folders/1qTxGaojM2eJn0SpHJl2Ujk4zvDkkadPR -O data

Processing file 1n7UewGksx3UZjn34-Q1QJtLXMMp42O_0 HAM10000_metadata.csv
Processing file 1sBEsgtYVo4X7APfg-_miZvpMttfS3eJ4 hnmist_28_28_RGB.csv


Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1n7UewGksx3UZjn34-Q1QJtLXMMp42O_0
To: c:\Users\rammu\Documents\projects\Deeplearning\practica\Deeplearning-\data\HAM10000_metadata.csv

  0%|          | 0.00/563k [00:00<?, ?B/s]
 93%|█████████▎| 524k/563k [00:00<00:00, 3.82MB/s]
100%|██████████| 563k/563k [00:00<00:00, 4.08MB/s]
Downloading...
From: https://drive.google.com/uc?id=1sBEsgtYVo4X7APfg-_miZvpMttfS3eJ4
To: c:\Users\rammu\Documents\projects\Deeplearning\practica\Deeplearning-\data\hnmist_28_28_RGB.csv

  0%|          | 0.00/91.8M [00:00<?, ?B/s]
  1%|          | 524k/91.8M [00:00<00:23, 3.84MB/s]
  2%|▏         | 1.57M/91.8M [00:00<00:15, 5.83MB/s]
  3%|▎         | 2.62M/91.8M [00:00<00:13, 6.40MB/s]
  4%|▍         | 3.67M/91.8M [00:00<00:13, 6.45MB/s]
  5%|▌         | 4.72M/91.8M [00:00<00:12, 6.92MB/s]
  6%|▋         | 5.77M/91.8M [00:00<00:1

## Pipeline de preprocesado

Esta celda centraliza todo el preprocesado. Reutilízala en los demás notebooks importando los arrays guardados, o cópiala directamente.

**Decisiones tomadas:**
- **Imputación de edad con mediana**: más robusta que la media ante outliers en datos médicos.
- **One-hot encoding** para sexo y localización: variables nominales sin orden implícito.
- **Normalización de edad** dividiendo por el máximo: mantiene la escala relativa.
- **Normalización de imágenes** dividiendo entre 255: lleva píxeles al rango [0, 1].
- **Stratify en el split**: garantiza que todas las clases estén representadas en train y test, crucial con el desbalanceo de HAM10000.

In [33]:
#Esta celda contiene todo el preproceamiento necesario para las imágenes y las etiquetas, y es la que copiaremos 

# Esta celda contiene todo el preprocesamiento necesario para las imágenes y las etiquetas
import os
import glob
import numpy as np
import cv2
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Cargar datos de imágenes desde CSV
path = r"../data/raw/hnmist_28_28_RGB.csv"
df = pd.read_csv(path)
print("Columnas del dataset:", df.columns[:10])

# Cargar metadata
metadata = pd.read_csv(r"../data/raw/HAM10000_metadata.csv")

# Etiquetas
y = metadata['dx'].values[:len(df)]
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_onehot = to_categorical(y_encoded)
num_classes = y_onehot.shape[1]

# Preprocesamiento tabular
tabular_data = metadata[['age', 'sex', 'localization']][:len(df)]
tabular_data = pd.get_dummies(tabular_data, columns=['sex', 'localization'], drop_first=True)
tabular_data['age'] = tabular_data['age'] / tabular_data['age'].max()
X_tab = tabular_data.values.astype(np.float32)

# Preprocesamiento imágenes
X_img = df.values.astype(np.float32).reshape(-1, 28, 28, 3)
X_img /= 255.0

# Dividir en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_img, y_onehot, test_size=0.2, random_state=42, stratify=y_encoded
)

print("Train set:", X_train.shape, y_train.shape)
print("Test set:", X_test.shape, y_test.shape)
print("Clases:", le.classes_)

Columnas del dataset: Index(['pixel0000', 'pixel0001', 'pixel0002', 'pixel0003', 'pixel0004',
       'pixel0005', 'pixel0006', 'pixel0007', 'pixel0008', 'pixel0009'],
      dtype='object')
Train set: (8012, 28, 28, 3) (8012, 7)
Test set: (2003, 28, 28, 3) (2003, 7)
Clases: ['akiec' 'bcc' 'bkl' 'df' 'mel' 'nv' 'vasc']
